In [ ]:
%%writefile submission.py
# ！！！PLEASE DO NOT EDIT IT ！！！
# This command will save the code you wrote as a submission.py file in the same directory, for the test script to read.


# You can IMPORT the package you need here.
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from typing import Optional, Tuple, Dict, Any, List
import os
from collections.abc import Iterable

# BELOW are MY answer functions

def linear_forward(weights, in_features):
    return torch.matmul(in_features, weights.T)

class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(vocab_size, d_model) * 0.02)
    
    def __call__(self, token_ids):
        return self.weight[token_ids]


class RMSNorm:
    def __init__(self, d_model, eps=1e-5):
        self.eps = eps
        self.weight = torch.ones(d_model)
    
    def __call__(self, x):
        original_dtype = x.dtype
        x_float = x.float()
        
        rms = torch.sqrt(torch.mean(x_float**2, dim=-1, keepdim=True) + self.eps)
        x_normalized = x_float / rms
        x_normalized = x_normalized.to(original_dtype)
        
        return x_normalized * self.weight

class SwiGLU:
    def __init__(self, d_model, d_ff):
        self.d_model = d_model
        self.d_ff = d_ff
        self.w1_weight = torch.randn(d_ff, d_model)
        self.w2_weight = torch.randn(d_model, d_ff) 
        self.w3_weight = torch.randn(d_ff, d_model)
    
    def __call__(self, x):
        w1_out = F.linear(x, self.w1_weight)
        w3_out = F.linear(x, self.w3_weight)
        swiglu_out = F.silu(w1_out) * w3_out
        return F.linear(swiglu_out, self.w2_weight)

def apply_rope(x, token_positions, d_k, theta=10000.0):
    if token_positions is None:
        seq_len = x.shape[-2]
        token_positions = torch.arange(seq_len, device=x.device).unsqueeze(0)

    freqs = 1.0 / (theta ** (torch.arange(0, d_k, 2, device=x.device).float() / d_k))

    angles = torch.einsum('i,j->ij', token_positions.float(), freqs)

    x_reshaped = x.view(*x.shape[:-1], -1, 2)
    cos_vals = torch.cos(angles).unsqueeze(-1)
    sin_vals = torch.sin(angles).unsqueeze(-1)
    
    x_rotated = torch.stack([
        x_reshaped[..., 0] * cos_vals - x_reshaped[..., 1] * sin_vals,
        x_reshaped[..., 0] * sin_vals + x_reshaped[..., 1] * cos_vals
    ], dim=-1)
    
    return x_rotated.view(*x.shape)

def scaled_dot_product_attention(Q, K, V, mask=None):
    if Q.dim() == 4:
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output
    else:
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        return torch.matmul(attn_weights, V)

def multihead_attention(d_model, num_heads, q_proj_weight, k_proj_weight, v_proj_weight, o_proj_weight, in_features):
    """多头注意力机制 - 调试版本"""
    print("=== Debug multihead_attention ===")
    batch_size, seq_len, d_in = in_features.shape
    d_k = d_model // num_heads
    d_v = d_model // num_heads
    
    print(f"Input: {in_features.shape}")
    print(f"Q weight: {q_proj_weight.shape}, K weight: {k_proj_weight.shape}")
    print(f"V weight: {v_proj_weight.shape}, O weight: {o_proj_weight.shape}")
    print(f"d_k: {d_k}, d_v: {d_v}")

    Q = F.linear(in_features, q_proj_weight)
    K = F.linear(in_features, k_proj_weight)
    V = F.linear(in_features, v_proj_weight)
    
    print(f"After linear - Q: {Q.shape}, K: {K.shape}, V: {V.shape}")

    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, d_v).transpose(1, 2)
    
    print(f"After reshape - Q: {Q.shape}, K: {K.shape}, V: {V.shape}")

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    print(f"Scores: {scores.shape}")

    mask = torch.triu(torch.ones(seq_len, seq_len, device=scores.device), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    scores = scores + mask.unsqueeze(0).unsqueeze(0)
    
    attn_weights = F.softmax(scores, dim=-1)
    attn_output = torch.matmul(attn_weights, V)
    print(f"Attention output: {attn_output.shape}")

    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    print(f"After concat heads: {attn_output.shape}")
 
    output = F.linear(attn_output, o_proj_weight)
    print(f"Final output: {output.shape}")
    print("=== End debug ===")
    
    return output

def apply_rope(x, positions, d_k, theta=10000.0):
    batch_size, seq_len = x.shape[0], x.shape[1]
    
    if positions is None:
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)

    assert d_k % 2 == 0, "d_k must be even for RoPE"

    freqs = 1.0 / (theta ** (torch.arange(0, d_k, 2, device=x.device).float() / d_k))

    angles = positions.unsqueeze(-1) * freqs.unsqueeze(0).unsqueeze(0)  # (batch_size, seq_len, d_k/2)

    x_reshaped = x.view(batch_size, seq_len, d_k // 2, 2)
    x1, x2 = x_reshaped[..., 0], x_reshaped[..., 1]
    
    cos_vals = torch.cos(angles)
    sin_vals = torch.sin(angles)
    
    x1_rotated = x1 * cos_vals - x2 * sin_vals
    x2_rotated = x1 * sin_vals + x2 * cos_vals

    x_rotated = torch.stack([x1_rotated, x2_rotated], dim=-1)
    return x_rotated.view(batch_size, seq_len, d_k)

def multihead_attention_with_rope(d_model, num_heads, max_seq_len, theta, q_proj_weight, k_proj_weight, 
                                        v_proj_weight, o_proj_weight, in_features, token_positions=None):
    batch_size, seq_len, d_in = in_features.shape
    d_k = d_model // num_heads
    d_v = d_model // num_heads

    assert q_proj_weight.shape == (num_heads * d_k, d_in), f"Q weight shape mismatch"
    assert k_proj_weight.shape == (num_heads * d_k, d_in), f"K weight shape mismatch"
    assert v_proj_weight.shape == (num_heads * d_v, d_in), f"V weight shape mismatch"
    assert o_proj_weight.shape == (d_model, num_heads * d_v), f"O weight shape mismatch"

    Q = F.linear(in_features, q_proj_weight)  # (batch_size, seq_len, num_heads * d_k)
    K = F.linear(in_features, k_proj_weight)  # (batch_size, seq_len, num_heads * d_k)
    V = F.linear(in_features, v_proj_weight)  # (batch_size, seq_len, num_heads * d_v)

    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len, d_k)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len, d_k)
    V = V.view(batch_size, seq_len, num_heads, d_v).transpose(1, 2)  # (batch_size, num_heads, seq_len, d_v)

    if token_positions is None:
        token_positions = torch.arange(seq_len, device=in_features.device).unsqueeze(0).expand(batch_size, seq_len)

    Q_rotated = []
    K_rotated = []
    
    for head in range(num_heads):
        Q_head = Q[:, head, :, :]  # (batch_size, seq_len, d_k)
        K_head = K[:, head, :, :]  # (batch_size, seq_len, d_k)
        
        Q_head_rotated = apply_rope(Q_head, token_positions, d_k, theta)
        K_head_rotated = apply_rope(K_head, token_positions, d_k, theta)
        
        Q_rotated.append(Q_head_rotated.unsqueeze(1))
        K_rotated.append(K_head_rotated.unsqueeze(1))
    
    Q = torch.cat(Q_rotated, dim=1)  # (batch_size, num_heads, seq_len, d_k)
    K = torch.cat(K_rotated, dim=1)  # (batch_size, num_heads, seq_len, d_k)

    d_k_float = torch.tensor(d_k, dtype=Q.dtype, device=Q.device)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(d_k_float)

    causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=scores.device), diagonal=1)
    causal_mask = causal_mask.masked_fill(causal_mask == 1, float('-inf'))
    scores = scores + causal_mask.unsqueeze(0).unsqueeze(0)
    
    attn_weights = F.softmax(scores, dim=-1)
    attn_output = torch.matmul(attn_weights, V)  # (batch_size, num_heads, seq_len, d_v)
 
    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, num_heads * d_v)

    output = F.linear(attn_output, o_proj_weight)  # (batch_size, seq_len, d_model)
    
    return output

class TransformerBlock:
    def __init__(self, d_model, num_heads, d_ff, max_seq_len, theta, weights):
        self.d_model = d_model
        self.num_heads = num_heads  
        self.d_ff = d_ff
        self.max_seq_len = max_seq_len
        self.theta = theta
        self.weights = weights
    
    def __call__(self, in_features):
        batch_size, seq_len, _ = in_features.shape

        ln1_weight = self.weights['ln1.weight']
        x_norm1 = self._apply_rmsnorm(in_features, ln1_weight)

        mask = torch.triu(torch.ones(seq_len, seq_len, device=in_features.device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf')).unsqueeze(0).unsqueeze(0)
        
        attn_output = self._multihead_attention_with_rope(x_norm1, mask)

        x_after_attn = in_features + attn_output

        ln2_weight = self.weights['ln2.weight']
        x_norm2 = self._apply_rmsnorm(x_after_attn, ln2_weight)

        ffn_output = self._apply_swiglu(x_norm2)

        x_after_ffn = x_after_attn + ffn_output
        
        return x_after_ffn
    
    def _apply_rmsnorm(self, x, weight, eps=1e-5):
        original_dtype = x.dtype
        x_float = x.float()
        
        rms = torch.sqrt(torch.mean(x_float**2, dim=-1, keepdim=True) + eps)
        x_normalized = x_float / rms
        x_normalized = x_normalized.to(original_dtype)
        
        return x_normalized * weight
    
    def _multihead_attention_with_rope(self, x, mask):
        batch_size, seq_len, _ = x.shape
        d_k = self.d_model // self.num_heads  
        d_v = self.d_model // self.num_heads  

        q_proj_weight = self.weights['attn.q_proj.weight']
        k_proj_weight = self.weights['attn.k_proj.weight']
        v_proj_weight = self.weights['attn.v_proj.weight']
        o_proj_weight = self.weights['attn.output_proj.weight']
        
        Q = torch.matmul(x, q_proj_weight.T)
        K = torch.matmul(x, k_proj_weight.T)
        V = torch.matmul(x, v_proj_weight.T)

        Q = Q.view(batch_size, seq_len, self.num_heads, d_k).transpose(1, 2)  # 使用self.num_heads
        K = K.view(batch_size, seq_len, self.num_heads, d_k).transpose(1, 2)  # 使用self.num_heads
        V = V.view(batch_size, seq_len, self.num_heads, d_v).transpose(1, 2)  # 使用self.num_heads

        token_positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        Q = self._apply_rope(Q, token_positions, d_k)
        K = self._apply_rope(K, token_positions, d_k)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        scores = scores + mask
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)

        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        output = torch.matmul(attn_output, o_proj_weight.T)
        
        return output
    
    def _apply_rope(self, x, positions, d_k, theta=10000.0):
        batch_size, num_heads, seq_len, _ = x.shape

        freqs = 1.0 / (theta ** (torch.arange(0, d_k, 2, device=x.device).float() / d_k))
        angles = positions.unsqueeze(-1) * freqs.unsqueeze(0).unsqueeze(0).unsqueeze(0)
        
        x_reshaped = x.view(batch_size, num_heads, seq_len, d_k // 2, 2)
        x1, x2 = x_reshaped[..., 0], x_reshaped[..., 1]
        
        cos_vals = torch.cos(angles)
        sin_vals = torch.sin(angles)
        
        x1_rotated = x1 * cos_vals - x2 * sin_vals
        x2_rotated = x1 * sin_vals + x2 * cos_vals
        
        x_rotated = torch.stack([x1_rotated, x2_rotated], dim=-1)
        return x_rotated.view(batch_size, num_heads, seq_len, d_k)
    
    def _apply_swiglu(self, x):
        w1_weight = self.weights['ffn.w1.weight']
        w2_weight = self.weights['ffn.w2.weight']
        w3_weight = self.weights['ffn.w3.weight']
        
        w1_out = torch.matmul(x, w1_weight.T)
        w3_out = torch.matmul(x, w3_weight.T)
        swiglu_out = F.silu(w1_out) * w3_out
        return torch.matmul(swiglu_out, w2_weight.T)

class TransformerLM:
    def __init__(self, vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta, weights):
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads 
        self.d_ff = d_ff
        self.rope_theta = rope_theta
        self.weights = weights
    
    def __call__(self, in_indices):
        batch_size, seq_len = in_indices.shape

        token_embeddings_weight = self.weights['token_embeddings.weight']
        x = token_embeddings_weight[in_indices] 

        for i in range(self.num_layers):
            layer_weights = {
                'attn.q_proj.weight': self.weights[f'layers.{i}.attn.q_proj.weight'],
                'attn.k_proj.weight': self.weights[f'layers.{i}.attn.k_proj.weight'],
                'attn.v_proj.weight': self.weights[f'layers.{i}.attn.v_proj.weight'],
                'attn.output_proj.weight': self.weights[f'layers.{i}.attn.output_proj.weight'],
                'ln1.weight': self.weights[f'layers.{i}.ln1.weight'],
                'ffn.w1.weight': self.weights[f'layers.{i}.ffn.w1.weight'],
                'ffn.w2.weight': self.weights[f'layers.{i}.ffn.w2.weight'],
                'ffn.w3.weight': self.weights[f'layers.{i}.ffn.w3.weight'],
                'ln2.weight': self.weights[f'layers.{i}.ln2.weight']
            }

            block = TransformerBlock(self.d_model, self.num_heads, self.d_ff, self.context_length, self.rope_theta, layer_weights)
            x = block(x)

        ln_final_weight = self.weights['ln_final.weight']
        x = self._apply_rmsnorm(x, ln_final_weight)

        lm_head_weight = self.weights['lm_head.weight']
        logits = torch.matmul(x, lm_head_weight.T)
        
        return logits
    
    def _apply_rmsnorm(self, x, weight, eps=1e-5):
        original_dtype = x.dtype
        x_float = x.float()
        
        rms = torch.sqrt(torch.mean(x_float**2, dim=-1, keepdim=True) + eps)
        x_normalized = x_float / rms
        x_normalized = x_normalized.to(original_dtype)
        
        return x_normalized * weight

def abs_topk_activation(in_features, k):
    if k <= 0:
        return torch.zeros_like(in_features)
    
    abs_x = torch.abs(in_features)
    values, _ = torch.topk(abs_x, k, dim=-1)
    threshold = values[..., -1].unsqueeze(-1)
    mask = abs_x >= threshold
    return torch.where(mask, in_features, torch.zeros_like(in_features))

def attention_with_sink(Q, K, V, sink_token=None, mask=None):
    print("=== Debug attention_with_sink ===")
    print(f"Q shape: {Q.shape}")
    print(f"K shape: {K.shape}")
    print(f"V shape: {V.shape}")
    if sink_token is not None:
        print(f"sink_token shape: {sink_token.shape}")
    if mask is not None:
        print(f"mask shape: {mask.shape}")
    
    if sink_token is not None:
        batch_size, num_heads, num_queries, head_dim = Q.shape
        num_keys = K.shape[2]
        
        if sink_token.dim() == 2:
            # sink_token: (1, head_dim) -> (batch_size, num_heads, 1, head_dim)
            sink_k = sink_token.unsqueeze(0).unsqueeze(0)  # (1, 1, 1, head_dim)
            sink_k = sink_k.expand(batch_size, num_heads, 1, head_dim)
        else:
            sink_k = sink_token
        
        print(f"sink_k shape after processing: {sink_k.shape}")
        
        sink_v = sink_k

        K_with_sink = torch.cat([sink_k, K], dim=2)  # (batch_size, num_heads, num_keys+1, head_dim)
        V_with_sink = torch.cat([sink_v, V], dim=2)  # (batch_size, num_heads, num_keys+1, head_dim)
        
        print(f"K_with_sink shape: {K_with_sink.shape}")
        print(f"V_with_sink shape: {V_with_sink.shape}")
        
        if mask is not None:
            sink_mask = torch.ones(batch_size, num_heads, num_queries, 1, 
                                 device=mask.device, dtype=mask.dtype)
            mask_with_sink = torch.cat([sink_mask, mask], dim=-1)  
            print(f"mask_with_sink shape: {mask_with_sink.shape}")
        else:
            mask_with_sink = None
        
        result = scaled_dot_product_attention(Q, K_with_sink, V_with_sink, mask_with_sink)
        print(f"Result shape: {result.shape}")
        print("=== End debug ===")
        return result
    else:
        return scaled_dot_product_attention(Q, K, V, mask)

def magnitude_pruning(model, sparsity_level):
    with torch.no_grad():
        for param in model.parameters():
            if param.dim() >= 2: 
                flat_weights = torch.abs(param.data.flatten())
                threshold = torch.quantile(flat_weights, sparsity_level)
                mask = torch.abs(param.data) > threshold
                param.data *= mask.float()  

def get_batch(dataset, batch_size, context_length, device):
    indices = torch.randint(0, len(dataset) - context_length, (batch_size,))
    inputs = torch.stack([torch.from_numpy(dataset[i:i+context_length]) for i in indices])
    targets = torch.stack([torch.from_numpy(dataset[i+1:i+context_length+1]) for i in indices])
    return inputs.to(device), targets.to(device)

def silu(in_features):
    return in_features * torch.sigmoid(in_features)

def softmax(in_features, dim):
    max_vals = torch.max(in_features, dim=dim, keepdim=True).values
    exp_vals = torch.exp(in_features - max_vals)
    return exp_vals / torch.sum(exp_vals, dim=dim, keepdim=True)

def cross_entropy(inputs, targets):
    log_probs = F.log_softmax(inputs, dim=-1)
    return F.nll_loss(log_probs, targets, reduction='mean')

def gradient_clipping(parameters, max_l2_norm):
    parameters = list(filter(lambda p: p.grad is not None, parameters))
    total_norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in parameters))
    clip_coef = max_l2_norm / (total_norm + 1e-6)
    if clip_coef < 1:
        for p in parameters:
            p.grad *= clip_coef

def get_lr_cosine_schedule(it, max_learning_rate, min_learning_rate, warmup_iters, cosine_cycle_iters):
    if it < warmup_iters:
        return max_learning_rate * (it / warmup_iters)
    else:
        progress = (it - warmup_iters) / cosine_cycle_iters
        return min_learning_rate + 0.5 * (max_learning_rate - min_learning_rate) * (1 + math.cos(math.pi * progress))

def save_checkpoint(model, optimizer, iteration, out):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'iteration': iteration
    }
    torch.save(checkpoint, out)

def load_checkpoint(src, model, optimizer):
    checkpoint = torch.load(src)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['iteration']     

Overwriting submission.py


In [2]:
import sys
import os
import pytest

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
os.chdir(project_root)

# Please CHANGE submission_dir to the directory of your answer.ipynb file, below is an example
submission_dir = os.path.join(project_root, "answers/yuanhao")
# submission_dir = os.path.join(project_root, "answers/xxxx")
if submission_dir not in sys.path:
    sys.path.insert(0, submission_dir)

# RUN TEST
# Args description:
# "-v": Verbose output
# "tests": Points to the test directory
# "-k sink": Run only tests related to "sink attention" (optional, for speed)
args = [
    "-v",
    "tests", 
    "-k", "test_attention_with_sink"
]

print(f"Current Working Directory: {os.getcwd()}")
pytest.main(args)

Current Working Directory: d:\juliannnnnn_project
============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- c:\Users\Julian\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: d:\juliannnnnn_project
plugins: anyio-4.7.0, jaxtyping-0.3.6
collecting ... collected 0 items

============================ no tests ran in 0.00s ============================


ERROR: file or directory not found: tests



<ExitCode.USAGE_ERROR: 4>